create widgets and variables for source_system, table_name, landing_timestamp, 
create a widget and variable for source_file_format

In [0]:
dbutils.widgets.text("source_system", "neon_postgres", "Source System")
dbutils.widgets.text("table_name", "hospitals", "Table Name")
dbutils.widgets.text("landing_timestamp", "2026-11-23_11:53:52", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "parquet", "Source File Format")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
dbutils.fs.ls("/Volumes/project1_gs_caresync/landing/landing_volume/neon_postgres/hospitals/2026-11-23_11:53:52/")

In [0]:
landing_path = f"/Volumes/project1_gs_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

# Check whether the landing path (and its parent directories) exists
try:
    dbutils.fs.ls(landing_path)
    path_exists = True
except Exception as e:
    if "FileNotFoundException" in str(e) or "does not exist" in str(e).lower():
        path_exists = False
    else:
        raise

if not path_exists:
    message = f"No new files received for {source_system}.{table_name} at {landing_timestamp}. Path not found: {landing_path}"
    print(message)
    dbutils.notebook.exit(message)

, if source_file_format is csv then read as csv else read as parquet. put simple if else statement 

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.csv(landing_path, header=True, inferSchema=True)
else:
    landing_df = spark.read.parquet(landing_path)

add 2 columns in landing_df, 1. landing_timestamp, 2.insert_timestamp as current timestamp

In [0]:
from pyspark.sql import functions as F

updated_df = landing_df.withColumn("landing_timestamp", F.lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", F.current_timestamp())

In [0]:
%skip
display(updated_df)

In [0]:
updated_df.write.mode("append").saveAsTable(f"project1_gs_caresync.bronze.{table_name}")

In [0]:
%skip
%sql
select * from project1_gs_caresync.bronze.hospitals